In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString

1. Import the data located at this link. It has information on people infected with dengue at the district level for 2015 to 2021.

In [ ]:
base_dengue = pd.read_csv(r'../../_data/data_dengue_peru.csv',thousands=',', na_values=['NaN', 'nan'], dtype={'Casos': 'float'})

2. Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: Use this code.

In [ ]:
# Create 'Ubigeo_Departamento', 'Ubigeo_Provincia' and rename 'Ubigeo' to 'Ubigeo_Distrito'
base_dengue['Ubigeo']=base_dengue['Ubigeo'].astype(str).str.zfill(6)
base_dengue['Ubigeo_Departamento'] = base_dengue['Ubigeo'].astype(str).str[:2]
base_dengue['Ubigeo_Provincia'] = base_dengue['Ubigeo'].astype(str).str[:4]
base_dengue = base_dengue.rename(columns={'Ubigeo':'Ubigeo_Distrito'})

In [ ]:
# Create data frames 'base_Distrito', 'base_Provincia', and 'base_Departamento' to facilitate the creation of the next plots

# Define the levels we are going to group the columns 'Casos'
nivel = ['Distrito', 'Provincia', 'Departamento']
# Dictionary to store the results
datas = {}
# Loop to create the data frames
for nivel in nivel:
    columna_ubigeo = f'Ubigeo_{nivel}'
    datas[nivel] = base_dengue.groupby([columna_ubigeo, 'Año', nivel ])['Casos'].sum(min_count=0).reset_index()
    
    exec(f"base_{nivel} = datas['{nivel}']")

In [ ]:
base_Distrito

In [ ]:
# Get geometry information
maps = gpd.read_file(r'../../_data/shape_file/DISTRITOS.shp')

In [ ]:
# Keeps geometry information for 'Distrito'
maps_DIST = maps[['IDDIST', 'geometry']]
maps_DIST = maps_DIST.rename({'IDDIST':'Ubigeo_Distrito'}, axis =1 )
maps_DIST['Ubigeo_Distrito'] = maps_DIST['Ubigeo_Distrito'].astype(str)

# Keeps geometry information for 'Provincia'
maps_PROV = maps[['IDPROV', 'geometry']]
maps_PROV = maps_PROV.rename({'IDPROV':'Ubigeo_Provincia'}, axis =1 )
maps_PROV['Ubigeo_Provincia'] = maps_PROV['Ubigeo_Provincia'].astype(str)

# Keeps geometry information for 'Departamento'
maps_DEP = maps[['IDDPTO', 'geometry']]
maps_DEP = maps_DEP.rename({'IDDPTO':'Ubigeo_Departamento'}, axis =1 )
maps_DEP['Ubigeo_Departamento'] = maps_DEP['Ubigeo_Departamento'].astype(str)

3. Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile.

In [ ]:
#Usamos la base filtrando el año 2021 y la renombramos

base_Distrito_2021 = base_Distrito[base_Distrito["Año"] == 2021]

In [ ]:
#Checamos que los valores en las columnas Ubigeo_Distrito son únicas para ambas bases

maps_DIST.Ubigeo_Distrito.is_unique
base_Distrito_2021.Ubigeo_Distrito.is_unique

In [ ]:
#Checamos que tipos de datos son los valores en ambas bases, con énfasis en Ubigeo_Distrito para hacer el merge

maps_DIST.dtypes
base_Distrito_2021.dtypes

In [ ]:
#Convertimos los valores de la columna Ubigeo_Distrito en la base_Distrito_2021 a string

base_Distrito_2021.loc[:, 'Ubigeo_Distrito'] = base_Distrito_2021['Ubigeo_Distrito'].astype(str)

In [ ]:
#Relizamos el merge, dejando todas las observaciones de la base maps_DIST, lo que nos genera valores NaN.

dataset = pd.merge(maps_DIST, base_Distrito_2021, how = "left", on = "Ubigeo_Distrito" )

In [ ]:
#Para poder observar los distintos valores únicos en la columna Casos se realiza una tabla de frecuencias

tabla_frecuencias_casos = dataset['Casos'].value_counts().reset_index()
tabla_frecuencias_casos.columns = ['Número de Casos', 'Frecuencia']
tabla_frecuencias_casos = tabla_frecuencias_casos.sort_values(by='Número de Casos')

tabla_frecuencias_casos

In [ ]:
!pip install --upgrade mapclassify

In [ ]:
#Se hace el mapeo considerando un colo diferente para los valores NaN

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(20, 20))
dataset.plot(ax = ax, 
        column='Casos', 
         cmap= 'Reds', 
         figsize=(20, 20), 
         linestyle='--',
         edgecolor='black', 
         legend = True,  
         scheme = "User_Defined", 
         missing_kwds= dict(color = "#DADADB",), 
         classification_kwds = dict( bins = [ 100, 200, 300, 400, 500, 1000, 1500, 2000,  2500, 3000 ] ), 
         legend_kwds=dict(  loc='upper left',
                            bbox_to_anchor=(1.01, 1),
                            fontsize='x-large',
                            title= "Número de casos por Distrito", 
                            title_fontsize = 'x-large', 
                            frameon= False )
            )

4. Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the province level.

In [ ]:
# import matplotlib.pyplot as plt

In [ ]:
# Create new dataframe by filtering province data by year = 2021

base_Prov_2021 = base_Provincia[base_Provincia["Año"] == 2021]
base_Prov_2021

In [ ]:
# Create new shapefile file where we aggregate district geometry at province level
maps_PROV1 = maps_PROV.dissolve( by = 'Ubigeo_Provincia' )
maps_PROV1.reset_index(drop=False, inplace=True) # Reset index to maintain 'Ubigeo_Provincia' column
# Show new province level geometry shapefiles
# fig, ax = plt.subplots( figsize = ( 12, 15 ) )
# maps_PROV1.plot( ax = ax )

In [ ]:
# Check for unique values in both geometry and values dataframes
print(maps_PROV1.Ubigeo_Provincia.is_unique)
print(base_Prov_2021.Ubigeo_Provincia.is_unique)
# Also, check 'Ubigeo_Provincia' column contains same type values on both dataframes
print(base_Prov_2021.dtypes)
print(maps_PROV1.dtypes)

In [ ]:
# Merge dataframes
prov_df = pd.merge(maps_PROV1, base_Prov_2021, how = "left", on = "Ubigeo_Provincia" )
prov_df

In [ ]:
# Check 'Casos' values frecuency

prov_casos = prov_df['Casos'].value_counts().reset_index()
prov_casos.columns = ['Número de Casos', 'Frecuencia']
prov_casos = prov_casos.sort_values(by='Número de Casos')

prov_casos

In [ ]:
# Plot map, consider value frecuency to establish bins for continous legend
fig, ax = plt.subplots(figsize=(20, 20))
prov_df.plot(ax = ax, 
        column='Casos', 
         cmap= 'Blues', 
         figsize=(20, 20), 
         linestyle='--',
         edgecolor='black', 
         legend = True,  
         scheme = "User_Defined", 
         missing_kwds= dict(color = "#DADADB",), 
         classification_kwds = dict( bins = [100, 250, 500, 1000, 1500, 2000, 2500] ), 
         legend_kwds=dict(  loc='upper left',
                            bbox_to_anchor=(1.01, 1),
                            fontsize='x-large',
                            title= "Número de casos por Provincia", 
                            title_fontsize = 'x-large', 
                            frameon= False )
            )

5. Use geopandas to plot the number of cases by the department for all the years using subplots. Every subplot for each year. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level.

In [ ]:
#Aggregate shapefile to department level
maps_DEP1 = maps_DEP.dissolve( by = 'Ubigeo_Departamento' )

# Get a dataframe with every year for each deparment
años = pd.DataFrame(base_Departamento['Año'].unique(), columns=['Año'])
dpto = maps.groupby(['IDDPTO', 'DEPARTAMEN']).size().reset_index(name='Freq')
dpto=dpto[['IDDPTO', 'DEPARTAMEN']]
dpto.columns=['Ubigeo_Departamento','Departamento']
departamento = pd.concat([dpto]*len(años), ignore_index=True)
años = pd.concat([años]*len(dpto), ignore_index=True)
tabla = pd.concat([departamento, años], axis=1)
df_Departamento = pd.merge(tabla, base_Departamento, how='left' ,on = ['Ubigeo_Departamento','Departamento','Año'] )

# Group cases by Cuartil
year=list(base_Departamento['Año'].unique())
for years in year:
    filtro = df_Departamento['Año'] == years  
    lista= list(map(str, list(df_Departamento.loc[filtro, 'Casos'].quantile([0.25, 0.5, 0.75,1]).astype(int)))) 
    df_Departamento.loc[filtro,'cat_casos'] = pd.cut(x=df_Departamento.loc[filtro, 'Casos'], 
                                                     bins=[0]+list(df_Departamento.loc[filtro, 'Casos'].quantile([0.25, 0.5, 0.75,1])),
                                                     labels=["Cuartil_1", "Cuartil_2", "Cuartil_3", "Cuartil_4"])

# Merge new base of cases and shapefile
departamento_df = pd.merge(maps_DEP1, df_Departamento, on = ['Ubigeo_Departamento'] )
departamento_df


In [ ]:
# Make a plot with number of cases as main data
fig, axes = plt.subplots(2, 4, figsize = ( 20, 20 ))
for cat, ax in zip(year, axes.flatten()):
    data = departamento_df[departamento_df["Año"] == cat]
    data.plot(column="Casos",
              legend=True, 
              edgecolor="black", 
              cmap="Oranges", 
              ax=ax, 
              linestyle='--', 
              missing_kwds= dict(color = "#DADADB",),
              categorical=False,
              #legend_kwds=dict(  loc='lower left'),
             )
    ax.set_title(cat)
    ax.axis("off")

fig.suptitle( 'Numbers of Cases, by Department', fontsize = 20 )
fig.tight_layout()

In [ ]:
# Make a plot with quartil of cases of each year as main data

fig, axes = plt.subplots(2, 4, figsize = ( 20, 20 ))
for cat, ax in zip(year, axes.flatten()):
    data = departamento_df[departamento_df["Año"] == cat]
    data.plot(column="cat_casos",
              legend=True, 
              edgecolor="black", 
              cmap="Oranges", 
              ax=ax, 
              linestyle='--', 
              missing_kwds= dict(color = "#DADADB",),
              categorical=False,
              legend_kwds=dict(  loc='lower left'),
             )
    ax.set_title(cat)
    ax.axis("off")
fig.suptitle( 'Numbers of Cases, by Department', fontsize = 20 )
fig.tight_layout()

6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.


In [ ]:
# we call the initial database
base_dengue

# Function to map weeks to quarters
def weeks_to_quarters(week):
    if week <= 13:
        return 'Q1'
    elif week <= 26:
        return 'Q2'
    elif week <= 39:
        return 'Q3'
    else:
        return 'Q4'

# Apply the function to create a new column 'Trimestres'
base_dengue['Trimestres'] = base_dengue['Semana'].apply(weeks_to_quarters)

base_dengue

# We group the number of cases by department location, department, year and quarters
base_dpto = base_dengue.groupby(['Ubigeo_Departamento', 'Departamento', 'Año', 'Trimestres'])['Casos'].sum(min_count=0).reset_index()


# We keep with observartions for year 2021
base_Departamento_2021 = base_dpto[base_dpto["Año"] == 2021]
base_Departamento_2021

In [ ]:
# Create new shapefile file where we aggregate district geometry at department level
maps_DEP1 = maps_DEP.dissolve( by = 'Ubigeo_Departamento' )
maps_DEP1.reset_index(drop=False, inplace=True)
maps_DEP1

In [ ]:
# Check for unique values in both geometry and values dataframes
print(maps_DEP1.Ubigeo_Departamento.is_unique)
print(base_Departamento_2021.Ubigeo_Departamento.is_unique)
# Also, check 'Ubigeo_Provincia' column contains same type values on both dataframes
print(base_Departamento_2021.dtypes)
print(maps_DEP1.dtypes)

In [ ]:
# Merge dataframes
dep_df = pd.merge(maps_DEP1, base_Departamento_2021, how = "left", on = "Ubigeo_Departamento" )
dep_df

In [ ]:
# We use a categorical legend with 5 bins
bins = pd.qcut( dep_df[ 'Casos' ], 5, retbins = True )[ 1 ][ 1: ]
type(bins)


In [ ]:
fig, axis = plt.subplots( nrows = 2, ncols = 2, figsize = ( 20, 20 ) )

idx = 0

for i in range( 2 ):
    for j in range( 2 ):
        ax = axis[ i ][ j ]
        cmap = plt.cm.Blues
        quarter = dep_df.Trimestres.unique()[ idx ]
        dg_temp = dep_df[ ( ( dep_df[ 'Trimestres' ] == quarter ) | ( dep_df[ 'Trimestres' ].isna() ) ) ]
        dg_temp.plot( column = 'Casos', cmap = cmap, linestyle = '--', edgecolor = 'black', ax = ax, legend = True, 
                      scheme = 'User_defined',
                      missing_kwds = dict( color = '#DADADB' ), 
                      classification_kwds = dict( bins = bins ), 
                      legend_kwds = dict( loc = 'upper left', bbox_to_anchor = ( 1.01, 1 ), title = 'Number of Cases', 
                                          frameon = False, fontsize = 'x-large', title_fontsize = 'x-large' ) )
        ax.get_xaxis().set_ticks([])
        ax.get_yaxis().set_ticks([])
        ax.set_title( f"Q{ str( quarter )[ 0 ] }" )
        ax.set_title( f"Q{idx+1}" )
        idx = idx + 1
fig.suptitle( 'Numbers of Cases per Quarter, by Region', fontsize = 20 )
fig.tight_layout()